In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import yaml
import os
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [ ]:
df = pd.read_excel("../data/cleaned/df_cleaned.xlsx")
df.head()

In [ ]:
df.describe(include='all').T

In [ ]:
# Separating Dependent and Independent features
X = df.drop(labels='COEMISSIONS', axis=1)
y = df["COEMISSIONS"]

In [ ]:
X.head()

In [ ]:
y

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_test.shape, y_test.shape

# Feature Engineering

In [ ]:
X_train.head()

## Encoding Categorical features
- From EDA we have observed that Fuel Consimption and Coemssion have shown almost linear correlation i.e. if fuel consumption increases then coemssion also increases.
- So, we can encode the categorical features according to their respective mean fuel consumption.

- The mapping dictinoary is saved for fututre reference at the time of deployment.
- If a new input comes from the user which is not saved in the dictionary should be imputed with glodal mean of the training dataset.

In [ ]:
CONFIG_FILEPATH="../config/column_config/"

In [ ]:
# Claculating global fuel mean
global_mean = X_train['FUEL CONSUMPTION'].mean()
global_mean

In [ ]:
data = {
    "global_mean": float(global_mean)
}

with open(CONFIG_FILEPATH+"global_mean.yaml", "w") as file:
    yaml.dump(data, file)

In [ ]:
model_mean_fuel_cons = X_train.groupby(by=['MODEL'])['FUEL CONSUMPTION'].mean().to_dict()
model_mean_fuel_cons

In [ ]:
X_train['MODEL'] = X_train['MODEL'].map(model_mean_fuel_cons)
X_train.head()

In [ ]:
make_mean_fuel_cons = X_train.groupby(by=['MAKE'])['FUEL CONSUMPTION'].mean().to_dict()
X_train['MAKE'] = X_train['MAKE'].map(make_mean_fuel_cons)
X_train.head()

In [ ]:
transmission_mean_fuel_cons = X_train.groupby(by=['TRANSMISSION'])['FUEL CONSUMPTION'].mean().to_dict()
X_train['TRANSMISSION'] = X_train['TRANSMISSION'].map(transmission_mean_fuel_cons)
X_train.head()

In [ ]:
fuel_type_mean_fuel_cons = X_train.groupby(by=['FUEL'])['FUEL CONSUMPTION'].mean().to_dict()
X_train['FUEL'] = X_train['FUEL'].map(fuel_type_mean_fuel_cons)
X_train.head()

## Saving the encoding mapping

In [ ]:
data_list = [model_mean_fuel_cons, make_mean_fuel_cons, transmission_mean_fuel_cons, fuel_type_mean_fuel_cons]
file_list = ['car_model_config.yaml', 'car_make_config.yaml', 'transmission_config.yaml', 'fuel_config.yaml']

for data, file in zip(data_list, file_list):
    with open(CONFIG_FILEPATH+file, "w") as file:
        yaml.dump(data, file)

## Visualizing the distribution of numerical columns

In [ ]:
plt.figure(figsize=(15, 6))
plt.suptitle('Univariate Analysis of Numerical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)

for i,col in enumerate(X_train.columns):
    plt.subplot(3, 3, i+1)
    sns.kdeplot(data=X_train[col],fill=True, color='r')
    plt.xlabel(col)
    plt.xticks(rotation=90)
    plt.tight_layout()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled

In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(2,1,1)
sns.kdeplot(data=X_train,fill=True)
plt.title("X_train before scaling")
plt.subplot(2,1,2)
sns.kdeplot(data=X_train_scaled,fill=True)
plt.title("X_train after scaling")
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
sns.boxplot(data=X_train)
plt.title("X_train before scaling")
plt.subplot(1,2,2)
sns.boxplot(data=X_train_scaled)
plt.title("X_train after scaling")
plt.show()

In [ ]:
sns.kdeplot(data=y_train,fill=True)

In [ ]:
y.skew()

### Feature encoding on test data:
- Test data should be encoded according to the mapping generated from the training dataset.

In [ ]:
# config loader
def config_loader(path=CONFIG_FILEPATH, file_list=None):
        all_configs = {} # Initializing empty disctionary
        keys = [i.split('_')[0] for i in file_list]
        for key, file in zip(keys, file_list):
                with open(path+file, "r") as f:
                        all_configs[key] = yaml.safe_load(f)
        return all_configs

In [ ]:
all_configs = config_loader(file_list=os.listdir(CONFIG_FILEPATH))
all_configs

In [ ]:
all_configs.keys()

In [ ]:
X_test.head()

In [ ]:
X_test.dtypes

In [ ]:
test_cat_col = [col for col in X_test.columns if X_test[col].dtypes=='str']
test_cat_col

In [ ]:
# Encode Model
def encode_model(x):
    if x in (all_configs['model'].keys()):
        return all_configs['model'][x]
    else:
        return all_configs['global']['global_mean']
    
X_test['MODEL'] = X_test['MODEL'].apply(lambda x: encode_model(x))
X_test.head()

In [ ]:
# Encode Make
def encode_model(x):
    if x in (all_configs['make'].keys()):
        return all_configs['make'][x]
    else:
        return all_configs['global']['global_mean']
    
X_test['MAKE'] = X_test['MAKE'].apply(lambda x: encode_model(x))
X_test.head()

In [ ]:
# Encode Transmission
def encode_model(x):
    if x in (all_configs['transmission'].keys()):
        return all_configs['transmission'][x]
    else:
        return all_configs['global']['global_mean']
    
X_test['TRANSMISSION'] = X_test['TRANSMISSION'].apply(lambda x: encode_model(x))
X_test.head()

In [ ]:
# Encode Transmission
def encode_model(x):
    if x in (all_configs['fuel'].keys()):
        return all_configs['fuel'][x]
    else:
        return all_configs['global']['global_mean']
    
X_test['FUEL'] = X_test['FUEL'].apply(lambda x: encode_model(x))
X_test.head()

In [ ]:
X_test.sample(20)

### Model Training

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = linear_pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

performance_matrix = pd.DataFrame({"Values":[mse,rmse,mae,r2]}, index=["MSE","RMSE","MAE","R2 Score"])
performance_matrix

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import RobustScaler, PowerTransformer

# 1️⃣ Build pipeline
pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),  # handles skewness
    ("scaler", RobustScaler()),                         # robust to outliers
    ("model", ElasticNet(max_iter=10000))
])

# 2️⃣ Define parameter grid
param_grid = {
    "model__alpha": np.logspace(-4, 1, 20),   # regularization strength
    "model__l1_ratio": np.linspace(0.1, 0.9, 9)  # L1 vs L2 mix
}

# 3️⃣ Grid search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=True
)

# 4️⃣ Fit
grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:", grid.best_params_)

In [ ]:
grid.feature_names_in_

In [ ]:
data = {}

for key, value in grid.best_params_.items(): # Updating best parameters
    data[key] = float(value)

data['inputs'] = list(grid.feature_names_in_) # Updationg input columns

ARTIFACT_PATH="../config/model_config/"

with open(ARTIFACT_PATH+"artifact_config.yaml", "w") as f:
    yaml.dump(data, f)

In [ ]:
y_pred = grid.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", rmse)

In [ ]:
r2_score(y_test, y_pred)

#### Assumption-1: scatter plot between y_test and y_pred should show linear pattern

In [ ]:
sns.jointplot(x=y_test, y=y_pred, kind='reg', line_kws={"color": "red"})
plt.xlabel("Actual Output")
plt.ylabel("Predicted Output")
plt.show()

#### Assumption-2: Density plot of residuals should be normaly distributed

In [ ]:
residuals = y_test - y_pred

# densityplot this residuals
sns.kdeplot(data=residuals,fill=True, color='g')
plt.show()

#### Assumption-3: scatter plot between y_pred and residuals should be unformly distributed i.e. no pattern

In [ ]:
# scatter plot with respect to prediction and residuals
plt.scatter(x=y_pred, y=residuals)
plt.xlabel("Predicted Output")
plt.ylabel("Residuals")
plt.show()

In [ ]:
import statsmodels.api as sm

model = sm.OLS(endog=y_train, exog=X_train).fit()
model.summary()

In [ ]:
import pickle

with open("../artifacts/model.pkl", "wb") as f:
    pickle.dump(grid, f)